In [1]:
import numpy as np
import scipy
from scipy.optimize import curve_fit
import math
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import pickle
plt = sns.mpl.pyplot
from scipy.optimize import fmin_l_bfgs_b


import h5py as h5

import json

# plotting the spec signal data at different temperatures:
import numpy as np
import scipy
from scipy import special

import sys

import ctypes

import time

from datetime import datetime
import pytz

import threading

import healpy as hp

import numpy as np
import matplotlib.pyplot as plt
import torch
#!pip install torch pyro-ppl matplotlib numpy
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS

import pyro.distributions.transforms as T

from pyro.infer.autoguide.initialization import init_to_value


/home/ecp/virtual_environment/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
wavelength = np.load( "/home/ecp/test_folder/wavelength_array.npy" )

In [4]:
wavelength

array([314.621, 317.34 , 320.056, 322.768, 325.478, 328.185, 330.889,
       333.589, 336.286, 338.98 , 341.671, 344.358, 347.042, 349.723,
       352.401, 355.075, 357.745, 360.412, 363.076, 365.736, 368.392,
       371.045, 373.694, 376.34 , 378.981, 381.619, 384.254, 386.884,
       389.511, 392.134, 394.753, 397.367, 399.979, 402.586, 405.189,
       407.788, 410.383, 412.973, 415.56 , 418.143, 420.721, 423.295,
       425.865, 428.431, 430.992, 433.549, 436.102, 438.65 , 441.194,
       443.733, 446.268, 448.798, 451.324, 453.845, 456.362, 458.874,
       461.381, 463.884, 466.382, 468.875, 471.364, 473.848, 476.327,
       478.801, 481.27 , 483.734, 486.194, 488.648, 491.098, 493.542,
       495.982, 498.416, 500.845, 503.27 , 505.689, 508.103, 510.512,
       512.915, 515.314, 517.707, 520.095, 522.477, 524.855, 527.227,
       529.593, 531.955, 534.31 , 536.661, 539.006, 541.345, 543.679,
       546.008, 548.331, 550.648, 552.96 , 555.266, 557.566, 559.861,
       562.151, 564.

### (This is the only part that needs to be run for actual Data Inference / Creating Database json files)

Now, that we have obtained everything from our calibration efforts, we can do the actual data inference for actual POCAM data. We do curve_fitting for the effective spectral profile and use the deduced parameters and uncertainties to finally infer the true POCAM parameters up to the associated/derived and propagated uncertainties.

In [1]:
# first we define some helper and fit functions

def simple_gaussian(x, mean, sigma, scaling):
    values = scaling * 1/(np.sqrt(2*np.pi * (sigma**2))) * np.exp(-((x-mean)**2)/(2*(sigma**2)))
    # * 1/(np.sqrt(2*np.pi * (sigma**2))) * 
    return(values)

def gaussian(x, mean, sigma, scaling, resolution):
    values = scaling * 1/(np.sqrt(2*np.pi * (resolution**2 + sigma**2))) * np.exp(-((x-mean)**2)/(2*(resolution**2 + sigma**2)))
    # * 1/(np.sqrt(2*np.pi * (resolution**2 + sigma**2))) *
    return(values)


def voigt(x, mu, sigma, gamma, amp):
    z = ((x - mu) + 1j * gamma) / (sigma * np.sqrt(2))
    return amp * np.real(wofz(z)) / (sigma * np.sqrt(2 * np.pi))


"""
def skewing_erf(x, mean, alpha):
    values = 1/2 * ( 1 + special.erf(alpha*(x-mean)/np.sqrt(2)) )
    return(values)

def skewed_gaussian(x, mean, sigma, scaling, alpha):
    values = 2*(gaussian(x, mean, sigma, scaling)*skewing_erf(x, mean, alpha))
    return(values)
"""

def wl_shift_model(x):
    """ This function takes a reconstructed wl [nm] and gives back the 'true' wl [nm] as modelled and calibrated 
        and the associated uncertainty [nm] for it as a tuple (true_wl, uncertainty). 
        The values can be trusted fairly within 370-530 [nm] range, outside they are severe victim to the 
        little amount of datapoints.
        The functions are modelled on 6 datapoints as a polynomial of degree 5.
        The datapoints for true wl were obtained by fitting normal distributions to 6 different FB/FL filter spectra.
        The recon wl values were deduced as mean values from pyro sampling. The errors were obtained by propagating the
        given errors on the true filter CWL and the std from the pyro sampling togeether to one eff. error."""
    a = 1.75676707e-10  #1.75699867e-10
    b = -4.24495837e-07 #-4.24526456e-07
    c = 4.06196466e-04 #4.06200416e-04
    d = -1.92362595e-01 #-1.92352130e-01
    e = 4.50686924e+01 #4.50633210e+01
    f = -4.17907302e+03 #-4.17830538e+03
    
    wl_shift = a*(x)**5 + b*(x)**4 + c*(x)**3 + d*(x)**2 + e*(x)**1 + f
    
    true_wl = x + wl_shift
    
    
    z = -9.45984094e-11 #-9.42721882e-11
    y = 2.19810338e-07 #2.19037942e-07
    w = -2.01225610e-04 #-2.00500912e-04
    v = 9.07280278e-02 #9.03910375e-02
    u = -2.01617985e+01 #-2.00840836e+01
    t = 1.77025995e+03 #1.76314469e+03
    
    err = np.abs( z*(x)**5 + y*(x)**4 + w*(x)**3 + v*(x)**2 + u*(x)**1 + t )
    
    return(true_wl, err)


In [2]:
class single_spec_data:

    def __init__(self,
                 hemisphere='39',
                 pwm=54000,
                 temp=-20,
                 coarse=1, fine=20,
                 mode='default',
                 diode='LMG405',
                 batch='batch2'):
        
        """ This class is to process single spec datasets. """

        self.diode = diode
        self.nominal_cwl = int(self.diode[-3:])
        self.hemisphere = hemisphere
        self.batch = batch
        
        self.pwm = pwm
        self.coarse = coarse
        self.fine = fine
        self.mode = mode
        self.temp = temp
        
        self.bg_cycles = 1
        self.signal_cycles = 1
        
        
        self.wavelength = np.load( "/home/ecp/test_folder/wavelength_array.npy" )
        
        
        #h5_file_path = f"/home/ecp/test_folder/prod_characterization_{self.hemisphere}/"
        h5_file_path = f"/home/ecp/disk1/pocam_data/{self.batch}/prod_characterization_{self.hemisphere}/"
        h = h5.File(h5_file_path+self.diode, 'r')        
    
        try:
            self.target = "1"
            if 'LMG' in diode:
                self.driver = 'lmg' + self.target
                if self.temp != "25C_precheck":
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)+'C'].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)+'C'].get('spec_bg') )
                else:
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)].get('spec_bg') )


            elif 'KAPU' in diode:
                self.driver = 'kapu' + self.target
                if self.temp != "25C_precheck":
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)+'C'].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)+'C'].get('spec_bg') )
                else:
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)].get('spec_bg') )
        except:
            self.target = "2"
            if 'LMG' in diode:
                self.driver = 'lmg' + self.target
                if self.temp != "25C_precheck":
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)+'C'].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)+'C'].get('spec_bg') )
                else:
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+str(self.coarse)+'-'+str(self.fine)+'/'+str(self.temp)].get('spec_bg') )


            elif 'KAPU' in diode:
                self.driver = 'kapu' + self.target
                if self.temp != "25C_precheck":
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)+'C'].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)+'C'].get('spec_bg') )
                else:
                    self.total_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)].get('spec_signal') )
                    self.bg_counts = np.array( h[self.driver+'/'+str(self.pwm)+'/'+self.mode+'/'+str(self.temp)].get('spec_bg') )
        
        
        # also extract the meas_time in unix time format:
        # datetime string in local time (Munich, Germany)
        if 'LMG' in diode:
            dt_str = h[self.driver+'/'+str(54000)+'/'+str(1)+'-'+str(20)+'/'+str(25)+'C'+'/'+'metadata'].attrs.get('datetime')  #'2025-04-13 22:12:41.830819'
        elif 'KAPU' in diode:
            dt_str = h[self.driver+'/'+str(54000)+'/'+'default'+'/'+str(25)+'C'+'/'+'metadata'].attrs.get('datetime')  #'2025-04-13 22:12:41.830819'
        date = dt_str.split(' ')[0]
        self.date = ''
        for ymd in date.split('-'):
            self.date += ymd
        #print(self.date)
        # Step 1: Parse the naive datetime (no timezone yet)
        dt_naive = datetime.strptime(dt_str, '%Y-%m-%d %H:%M:%S.%f')
        # Step 2: Localize to Europe/Berlin timezone (handles DST)
        munich_tz = pytz.timezone('Europe/Berlin')
        dt_local = munich_tz.localize(dt_naive)
        # Step 3: Convert to UTC
        dt_utc = dt_local.astimezone(pytz.utc)
        # Step 4: Convert to Unix timestamp
        unix_time = dt_utc.timestamp()
        self.meas_time = unix_time
        #print("Unix time:", unix_time)
        
        
        # eventually properly close the h5 file:
        h.close()
                 
            
            
        # we want to combine the several bg-measurements to a single effective one by averaging over all of them
        # (just in case self.bg_cycles is not 1):
        self.average_bg_counts = np.zeros(288)
        for i in range(0, self.bg_cycles):
            self.average_bg_counts += self.bg_counts[i] / self.bg_cycles

        # also we want to give some info on the distribution through the several cycles, i.e. the std for every wavelength
        # (mind we have to do another/separate for loop as we are using the final result of the loop above):
        self.std_bg_counts = np.zeros(288)
        for i in range(0, self.bg_cycles):
            self.std_bg_counts += (self.bg_counts[i] - self.average_bg_counts)**2
            self.std_bg_counts = np.sqrt(self.std_bg_counts) / self.bg_cycles

        # now get rid of the statistical bg to obtain pure signal:
        self.signal_counts = []
        # Mind: we have to be aware that we don't just set the initial object self.signal_counts = self.total_counts
        # as we will change the actual self.total_counts raw data itself !
        # (compare to: mutable objects and where their changes are made/stored to !)

        # (again just in case self.signal_cycles is not 1):
        for i in range(0, len(self.total_counts)):
            self.signal_counts.append(self.total_counts[i] - self.average_bg_counts)
        self.signal_counts = np.array( self.signal_counts )

        # then we want to combine the several signal-measurements,
        # where we subtracted the average bg-counts from the respective total-counts,
        # to a single effective one by averaging over all of them:
        self.average_signal_counts = np.zeros(288)
        for i in range(0, self.signal_cycles):
            self.average_signal_counts += self.signal_counts[i] / self.signal_cycles

        # also we want to give the std for every result of wavelength bin, respectively:
        self.std_signal_counts = np.zeros(288)
        for i in range(0, self.signal_cycles):
            self.std_signal_counts += (self.signal_counts[i] - self.average_bg_counts)**2
            self.std_signal_counts = np.sqrt(self.std_signal_counts) / self.signal_cycles

        

        self.norm = 1 / np.sum(self.average_signal_counts)
        self.average_signal_counts *= self.norm
        self.bg_std = np.std( self.average_bg_counts ) * self.norm
        
        
        self.y_err = np.sqrt( np.abs(self.average_signal_counts) )
                
        
        
        
        
        self.peak = np.max(self.average_signal_counts)
        peak_index = np.where(self.average_signal_counts == self.peak)[0][0]
        self.peak_wl = self.wavelength[peak_index]
        
        # interpolate the shape of the pulse detected by the PMT:
        self.interpolation = scipy.interpolate.InterpolatedUnivariateSpline(self.wavelength, self.average_signal_counts)
        
        interpolation_wl = np.linspace(self.peak_wl-30,self.peak_wl+30,100)
        interpolation_data = self.interpolation(interpolation_wl)
        
        left_wl = interpolation_wl[interpolation_wl < self.peak_wl]
        left_data = interpolation_data[:len(left_wl)]
        left_index = [ i_ for i_ in range(len(left_wl)) if (np.abs(left_data[i_]-0.5*self.peak))==np.min(np.abs(left_data-0.5*self.peak)) ][0]
        self.start = left_wl[left_index]
        
        right_wl = interpolation_wl[interpolation_wl > self.peak_wl]
        right_data = interpolation_data[-len(right_wl):]
        right_index = [ i_ for i_ in range(len(right_wl)) if (np.abs(right_data[i_]-0.5*self.peak))==np.min(np.abs(right_data-0.5*self.peak)) ][0]        
        self.end = right_wl[right_index]

        approx_width = self.end-self.start
        
        
        self.approx_eff_width = approx_width
        self.approx_eff_std = self.approx_eff_width/2.3548
        

        self.scaling_initial_guess = 1*np.max(self.average_signal_counts) * np.sqrt(2*math.pi) * self.approx_eff_std**2
        
        
        
        
        
        # Pyro inferred best values and errors for spectrometer rersolution:
        if "365" in self.diode:
            self.resolution_spec = 4.11
            self.resolution_spec_err = 0.98
        if "LMG405" in self.diode:
            self.resolution_spec = 2.80
            self.resolution_spec_err = 0.77
        if "KAPU405" in self.diode:
            self.resolution_spec = 2.80
            self.resolution_spec_err = 0.77
        if "450" in self.diode:
            self.resolution_spec = 2.48
            self.resolution_spec_err = 0.78
        if "465" in self.diode:
            self.resolution_spec = 2.66
            self.resolution_spec_err = 0.82
        if "520" in self.diode:
            self.resolution_spec = 3.31
            self.resolution_spec_err = 0.05
            
        
        
        
        
        
        fit_xdata = self.wavelength[self.wavelength > (self.nominal_cwl-20)]
        fit_ydata = self.average_signal_counts[self.wavelength > (self.nominal_cwl-20)]
        fit_ydata = fit_ydata[fit_xdata < (self.nominal_cwl+20)]
        fit_xdata = fit_xdata[fit_xdata < (self.nominal_cwl+20)]
        


        fit, cov = curve_fit(simple_gaussian,
                             fit_xdata,
                             fit_ydata,
                             p0=[self.nominal_cwl,
                                 self.approx_eff_std,
                                 self.scaling_initial_guess
                                ],
                             bounds=([self.nominal_cwl-30,
                                      0,
                                      0
                                     ],
                                     [self.nominal_cwl+30,
                                      np.inf,
                                      np.inf
                                     ]),
                             maxfev=10000)
        
        self.fit = fit
        self.cov = cov
        
        
        #print(self.fit)
        #print(self.cov)
        
        
        self.fit_function = lambda x: simple_gaussian(x, *fit)
        """
        self.fit_function = lambda x: skewed_gaussian(x, *fit)
        mean=fit[0], sigma=fit[1], lamb=fit[2], scale=fit[3], offset=fit[4]
        """
        
        self.cwl = self.fit[0]
        self.cwl_err = np.sqrt( np.abs(self.cov[0][0]) )
        
        self.eff_std = self.fit[1]
        self.eff_std_err = np.sqrt( np.abs(self.cov[1][1]) )
        self.eff_fwhm = 2.3548*self.eff_std
        self.eff_fwhm_err = 2.3548*self.eff_std_err
        
        self.eff_scaling = self.fit[2]
        self.eff_scaling_err = np.sqrt( np.abs(self.cov[2][2]) )
        
        
        
        self.pocam_cwl, self.wl_shift_err = wl_shift_model( self.cwl )
        self.pocam_cwl_err = np.sqrt( self.wl_shift_err**2 + self.cwl_err**2 )
        self.pocam_std = np.sqrt( self.eff_std**2 - self.resolution_spec**2 )
        self.pocam_std_err = np.sqrt( (self.eff_std/self.pocam_std)**2*self.eff_std_err**2 + (self.resolution_spec/self.pocam_std)**2*self.resolution_spec_err**2 )
        self.pocam_fwhm = 2.3548*self.pocam_std
        self.pocam_fwhm_err = 2.3548*self.pocam_std_err
        
        
        
        
        
            
       
    
        
        
        fig = plt.figure(figsize=(16, 9))
        ax = fig.add_subplot(1, 1, 1)

        cwl = self.cwl
        pocam_cwl = self.pocam_cwl
        pocam_fwhm = self.pocam_fwhm
        eff_fwhm = self.eff_fwhm
        average_signal_counts = self.average_signal_counts
        nominal_cwl = self.nominal_cwl
        fit_function = self.fit_function
        
        wavelength = self.wavelength
        

        
        plt.scatter(wavelength, average_signal_counts, color='blue', marker='o', s=20,
                    label=f'Recon CWL: {round(cwl,2)} nm \n POCAM CWL: {round(pocam_cwl,2)} nm \nEff. FWHM: {round(eff_fwhm,2)} nm \nPOCAM FWHM: {round(pocam_fwhm,2)} nm')#\nFWHM: {round(fwhm,3)} nm')
        
        self.fit_x_array = np.linspace(nominal_cwl-50,nominal_cwl+50,1001)
        self.fit_y_array = fit_function(np.linspace(nominal_cwl-50,nominal_cwl+50,1001))
        plt.plot(self.fit_x_array,
                 self.fit_y_array,
                 color='black', linestyle='--', alpha=1, linewidth=0.7,
                 label=f"")
        
        plt.plot(np.linspace(nominal_cwl-50,nominal_cwl+50,1001),
                 self.interpolation(np.linspace(nominal_cwl-50,nominal_cwl+50,1001)),
                 color='orange', linestyle='--', alpha=1, linewidth=0.7,
                 label=f"")

        plt.xlabel("Wavelength [nm]", fontsize=27, labelpad=9)
        plt.ylabel("Relative Signal Counts", fontsize=27, labelpad=9)
        plt.xlim(nominal_cwl-30, nominal_cwl+35)
        plt.ylim(-0.005, max( 1.2*np.max(fit_function(np.linspace(nominal_cwl-40,nominal_cwl+30,6000))), 1.2*np.max(self.average_signal_counts) ) )
                 
        ax.tick_params(axis='both', labelsize=17, pad=8)

        plt.legend(loc='upper right', fontsize=18)

        plt.grid(True, alpha=0.35)
        """
        try:
            plt.savefig(f"/home/ecp/test_folder/Database_calibrations/Spec_plots/curve_fit_hem_{self.hemisphere}_{self.diode}.png", dpi=700, bbox_inches='tight')
        except Exception as plot_err:
            print(plot_err)
        """    
        plt.show()
        plt.close() 
        
        
        
        
        
        
        
        
        
        #print()
        #print("Data - bg-subtracted and normalized")
        #print("Wavelength array: ", wavelength)
        #print("Signal Counts: ", average_signal_counts)
        
        
        
        print()
        print("Fit")
        print("CWL: ", round( self.cwl ,2), f" (+/- {round(self.cwl_err,2)})")
        print("Eff. FWHM: ", round( self.eff_fwhm ,2), f" (+/- {round(self.eff_fwhm_err,2)})")
        print("Eff. Scaling: ", round( self.eff_scaling ,2), f" (+/- {round(self.eff_scaling_err,2)})")
        #print("Fit-wavelength array: ", np.linspace(nominal_cwl-50,nominal_cwl+50,1001))
        #print("Fit-Signal array: ", fit_function(np.linspace(nominal_cwl-50,nominal_cwl+50,1001)))
        
        

        print()
        print("POCAM")
        print("POCAM CWL: ", round( self.pocam_cwl ,2), f" (+/- {round(self.pocam_cwl_err,2)})")
        print("POCAM STD: ", round( self.pocam_std ,2), f" (+/- {round(self.pocam_std_err,2)})")
        print("POCAM FWHM: ", round( self.pocam_fwhm ,2), f" (+/- {round(self.pocam_fwhm_err,2)})")
        


In [3]:
outfile = '/home/ecp/drop_operation/dev_hem_batch_dic.json'

with open(outfile, 'r') as file:
    devices = json.load(file)

In [ ]:


pocam_device_number = list(devices.keys())

hemisphere = None # devices[pocam_device_number][0]
batch = device[pocam_device_number][1]

emitters = ['LMG365', 'LMG405', 'LMG450', 'LMG520', 'KAPU405', 'KAPU465']


if 'LMG' in emitter:
    driver = 'l'               # 'l' for lmg
else:
    driver = 'k'               # 'k' for kapu


power = ['10000', '15000', '20000', '25000', '30000', '35000', '45000', '54000', '7500']

lmg_widths = ['1-10', '1-20', '1-40', '2-20']

kapu_widths = ['default', 'fast']

temp = ['-10C', '-20C', '-30C', '-40C', '0C', '25C', '25C_precheck']

In [14]:
hemisphere = '11'

emitter = 'LMG405'             # our standard form of 'LMG405', 'KAPU465', etc.
if 'LMG' in emitter:
    driver = 'l'               # 'l' for lmg
else:
    driver = 'k'               # 'k' for kapu
pocam_device_number = '003'    # needs to be of the form '001', '002', ... , '019', etc.
#target = 'master'             # 'master' or 'slave'

temp = -10    

coarse = 1
fine = 20
mode = 'default'

power = 54000
batch = 'batch1'





# Setting up the dict to be stored as json file:
spectral_distribution = {}     # this is the dict to be stored as the Database json file for the spectral data for one L(E)D


spectral_distribution["device_uid"] = ''        # "pocam-20240405_001"
spectral_distribution["subdevice_uid"] = ''     # "pocam-led-{target}_{driver}-{emitter[-3:]}_{pocam_device_number}"
spectral_distribution["meas_name"] = "led-spectral-profile"
spectral_distribution["meas_class"] = "display"
spectral_distribution["meas_stage"] = "calibration"
spectral_distribution["meas_group"] = "spectral-profile"
spectral_distribution["meas_site"] = "tum"
spectral_distribution["meas_time"] = None        # 1717684138.9665911 , in uinx time format
spectral_distribution["meas_data"] = []




### start loop here ###



# Running the actual data inference (and plotting, etc.):
spec = single_spec_data(hemisphere=hemisphere,
                         pwm=power,
                         temp=temp,
                         coarse=coarse, fine=fine,
                         mode=mode,
                         diode= emitter,
                         batch= batch)  




# specifying some more key-value pairs for identification etc.:
spectral_distribution["device_uid"] = f"pocam-{spec.date}_{pocam_device_number}"
if spec.target == '1':
    target = 'master'
elif spec.target == '2':
    target = 'slave'
spectral_distribution["subdevice_uid"] = f"pocam-led-{target}_{driver}-{emitter[-3:]}_{pocam_device_number}"
spectral_distribution["meas_time"] = spec.meas_time




# sub-dictionaries to dump data in them:
# for each single spectral profile, specified by the used POCAM settings and the temp.,
# we have 3 sub-dict, one "graph-with-fit" for the spectral profile, one "value" for its CWL and one "value" for its FWHM


# "graph-with-fit":
graph_with_fit = {}
graph_with_fit["data_format"] = "graph-with-fit"
if 'LMG' in spec.diode:
    graph_with_fit["coarse"] = spec.coarse
    graph_with_fit["fine"] = spec.fine
elif 'KAPU' in spec.diode:
    graph_with_fit["mode"] = spec.mode                            # str: either 'default' or 'fast'
graph_with_fit["power"] = spec.pwm                                # integer
graph_with_fit["temperature"] = spec.temp                         # number (pos. or neg.), e.g. 25, 0, -20, etc.
graph_with_fit["x_label"] =  "Wavelength [nm]"
graph_with_fit["y_label"] =  "Relative Counts"
graph_with_fit["x_values"] = spec.wavelength.tolist()             # array of wavelength 'stamps'/values at which spec measured
graph_with_fit["y_values"] = spec.average_signal_counts.tolist()  # array of normalized and bg-subtracted data
graph_with_fit["x_min"] = np.min(spec.wavelength)
graph_with_fit["x_max"] = np.max(spec.wavelength)
graph_with_fit["n_bins"] = len(spec.wavelength)
graph_with_fit["fit_x_min"] = np.min(spec.fit_x_array)
graph_with_fit["fit_x_max"] = np.max(spec.fit_x_array)
graph_with_fit["fit_n_points"] = len(spec.fit_x_array)
graph_with_fit["fit_y_values"] = spec.fit_y_array.tolist()
graph_with_fit["title"] = "Spectral Profile"
spectral_distribution["meas_data"].append(graph_with_fit)



# "value" FWHM:
value_fwhm = {}
value_fwhm["data_format"] = "value"
value_fwhm["value"] = round( spec.pocam_fwhm , 2 )
value_fwhm["error"] = round( spec.pocam_fwhm_err , 2 )
if 'LMG' in spec.diode:
    value_fwhm["coarse"] = spec.coarse
    value_fwhm["fine"] = spec.fine
elif 'KAPU' in spec.diode:
    value_fwhm["mode"] = spec.mode
value_fwhm["power"] = spec.pwm
value_fwhm["temperature"] = spec.temp
value_fwhm["label"] = "FWHM"
value_fwhm["title"] = "Full-Width-Half-Maximum [nm]"
spectral_distribution["meas_data"].append(value_fwhm)



# "value" CWL:
value_cwl = {}
value_cwl["data_format"] = "value"
value_cwl["value"] = round( spec.pocam_cwl , 2 )
value_cwl["error"] = round( spec.pocam_cwl_err , 2 )
if 'LMG' in spec.diode:
    value_cwl["coarse"] = spec.coarse
    value_cwl["fine"] = spec.fine
elif 'KAPU' in spec.diode:
    value_cwl["mode"] = spec.mode
value_cwl["power"] = spec.pwm
value_cwl["temperature"] = spec.temp
value_cwl["label"] = "CWL"
value_cwl["title"] = "Central-Wavelength [nm]"
spectral_distribution["meas_data"].append(value_cwl)





### end loop here ###





# We further add some comments and support files (links) to the json file:

# the comments are describing the data but can be simply copy-pasted for all different L(E)Ds,
# as they are not specifying on the exact data values:
spectral_distribution["comments"] = [
                                        "The data shows the spectral profile/distribution of the light emission for the chosen L(E)D, operated at the specified parameters/conditions (temperature, applied voltage, pulse shape settings, etc.)",
                                        "The experimental data points give the spectral profile at some certain discrete wavelength values, determined by the used spectrometer.",
                                        "The experimental data is normalized such that the sum of all single datapoints (Relative (Signal) Counts) is equal to 1",
                                        "coarse and fine (for lmg-driven L(E)Ds), or mode (for kapu-driven L(E)Ds) are referring to the used pulse shape / time profile settings. ",
                                        "power is the numerical Pulse-Width-Modulation (PWM) value that is set to apply voltage, which handles emission intensities. (PWM is integer of 0 - 54000 corresponding to ~ 0 - 32 V)",
                                        "The experimental data shows true normalized signal counts, where background has already been subtracted",
                                        "(If available) A fit function that can describe the sepctral profile, based on the experimental data, is included in 'graph-with-fit'. The fit function is a simple Gaussian distribution (mu, sigma) multiplied by some (unphysical) scaling factor.",
                                        "Mind that the fit-function is not yet to be used as true Probability-Density-Function (PDF), as its integral is not normalized to 1.",
                                        "Mind also that the derived / inferred values for the FWHM and CWL of the true L(E)D spectrum do not correspond presicley to the values one might guess / see when plotting the experimental data and the fir-function, since the true spectral profile is object of wavelength-shifting and resolution (broadening) effects of the spectrometer, which were already carefully respected to derive true POCAM parameters. Thus to reconstruct true L(E)D spectral distribution, you may want to plot a Gaussian based on the specifically given CWL and FWHM values (and some unphysical scaling factor, depending on imposed normalization requirements). Since the spectral broadening is the effect of a convolution of a normalized Gaussian with the (also Gaussian) true POCAM spectral profile, one can simply take the same scaling factor used for the fit-function (deducable from the height) and end up with the same normalization used in the fit (which is determined by requiring that the sum of all experimental data points = relative counts is equal to 1)",
                                        "(If available from fitting) The FWHM of the spectral profile at the given parameters and conditions is stored as a value, including its uncertainty",
                                        "(If available from fitting) The CWL of the spectral profile at the given parameters and conditions is stored as a value, including its uncertainty",
                                        "Structure of the device_uid: 'pocam-{date of measurement}_{pocam device number}",
                                        "Structure of the subdevice_uid: 'pocam-led_{target side}_{pulse driver}-{nominal emission wavelength}_{pocam device number} , where target side is either master (target 1) or slave (target 2) and pulse driver is either l (for lmg) or k (for kapu)"
                                    ]


# the support files contain one link where a POCAM_documentation file shall be uploaded,
# explaining data taking, processing and interpretation in a more detailed way
# but also contains the link to the specific raw data file(s) that is specified by the selected L(E)D
spectral_distribution["support_files"] = [
                                            {"filetype": "hdf5",
                                             "hostname": "data.icecube.wisc.edu",
                                             "pathname": f"/data/exp/IceCubeUpgrade/commissioning/pocam/pocam_{pocam_device_number}/{target}_hemisphere/{emitter}",
                                             "comment" : "Here you can find the raw data, stored as an hdf5 file. For more detailed info on the structure and interpretation of the data files please consult the POCAM documentation guide."
                                            },
                                            {"filetype": "pdf",
                                                 "hostname": "data.icecube.wisc.edu",
                                                 "pathname": "/data/exp/IceCubeUpgrade/commissioning/pocam/POCAM_documentation.pdf",
                                                 "comment" : "Here you can find the POCAM documentation guide, for more detailed info on the data taking, data processing, interpretation, etc.."
                                                }
                                    ]


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [6]:
spectral_distribution

{'device_uid': 'pocam-20250218_016',
 'subdevice_uid': 'pocam-led-master_l-405_016',
 'meas_name': 'led-spectral-profile',
 'meas_class': 'display',
 'meas_stage': 'calibration',
 'meas_group': 'spectral-profile',
 'meas_site': 'tum',
 'meas_time': 1739892629.026906,
 'meas_data': [{'data_format': 'graph-with-fit',
   'coarse': 1,
   'fine': 20,
   'power': 54000,
   'temperature': -10,
   'x_label': 'Wavelength [nm]',
   'y_label': 'Relative Counts',
   'x_values': [314.621,
    317.34,
    320.056,
    322.768,
    325.478,
    328.185,
    330.889,
    333.589,
    336.286,
    338.98,
    341.671,
    344.358,
    347.042,
    349.723,
    352.401,
    355.075,
    357.745,
    360.412,
    363.076,
    365.736,
    368.392,
    371.045,
    373.694,
    376.34,
    378.981,
    381.619,
    384.254,
    386.884,
    389.511,
    392.134,
    394.753,
    397.367,
    399.979,
    402.586,
    405.189,
    407.788,
    410.383,
    412.973,
    415.56,
    418.143,
    420.721,
   